# TinyLeafFormer vs parameter-matched students on PlantDoc (seed 42, saved teacher)

**Colab notebook C1 of the PlantDoc set.** This notebook trains only: **CNN-only, param-matched**.

* Runtime > Change runtime type > **T4 GPU**, then **Run all**. Keep the tab open.
* Upload `teacher.pth` once to `MyDrive/tinyleafformer/` (the same file as on Kaggle).
* Results go to `MyDrive/tinyleafformer/plantdoc/results` (shared by the whole set). If Colab
  disconnects, open the notebook again and Run all: finished runs are skipped.
* Run **A1 first** (it contains `Full`, the reference row and the TinyLeafFormer row of the
  comparison). The others can run in any order. To run several at the same time from different
  Google accounts, share the `tinyleafformer` folder with them and add a shortcut to My Drive.
* All rows of the set must be run on Colab (not mixed with Kaggle runs), with `NW = 2`.

### Shared setup (copied from the main notebook)
Configuration, data index, background randomization, `DualResDataset`, architecture, losses
and `train_da_epoch` are taken verbatim from the main notebook, with three small patches:
the background-randomization RNG is seeded (it used wall-clock time) and `train_da_epoch` gets a
`kd_on_target` switch.

In [ ]:
# ============================================================
# COLAB SETUP: GPU, Google Drive, datasets, what this notebook runs
# ============================================================
# Every notebook of this set writes into the SAME Drive folder, each to its own runs_<tag>.csv,
# so the notebooks can run one after another (or in parallel) without overwriting each other.
NB_TAG   = "C1"
RUN_ONLY = ['CNN-only, param-matched']

import os, glob, zipfile, shutil, subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
      or "!! NO GPU -- Runtime > Change runtime type > T4 GPU")

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/tinyleafformer"          # shared by all notebooks of this set
os.makedirs(DRIVE_DIR, exist_ok=True)

# Teacher: upload teacher.pth (the same file used on Kaggle) to this Drive folder once.
TEACHER_FILE_DRIVE = f"{DRIVE_DIR}/teacher.pth"
assert os.path.exists(TEACHER_FILE_DRIVE), f"Upload the teacher file to {TEACHER_FILE_DRIVE}"


def find_dir(root, name):
    for dp, dns, _ in os.walk(root):
        for d in dns:
            if d.lower() == name.lower():
                return os.path.join(dp, d)
    return None


# Datasets are downloaded to the local Colab disk (fast) with kagglehub. If it asks for
# credentials: kaggle.com > Settings > API > Create New Token, then enter username and key.
import kagglehub
PV_ROOT  = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
PD_ROOT  = kagglehub.dataset_download("nirmalsankalana/plantdoc-dataset")
PV_COLOR = find_dir(PV_ROOT, "color")
PV_SEG   = find_dir(PV_ROOT, "segmented")
assert PV_COLOR and PV_SEG, f"PlantVillage color/segmented folders not found under {PV_ROOT}"
print("PlantVillage color     :", PV_COLOR)
print("PlantVillage segmented :", PV_SEG)
print("PlantDoc               :", PD_ROOT)
print(f"\nThis notebook ({NB_TAG}) runs: {RUN_ONLY}")


GPU 0: Tesla T4 (UUID: GPU-dd3d6910-652d-7f33-bef8-e5a816aaff33)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Using Colab cache for faster access to the 'plantdoc-dataset' dataset.
PlantVillage color     : /kaggle/input/plantvillage-dataset/plantvillage dataset/color
PlantVillage segmented : /kaggle/input/plantvillage-dataset/plantvillage dataset/segmented
PlantDoc               : /kaggle/input/plantdoc-dataset

This notebook (C1) runs: ['CNN-only, param-matched']


In [ ]:
# ============================================================
# PART 0: IMPORTS AND CONFIGURATION
# ============================================================
import os, sys, time, random, warnings, copy, json, glob, math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter
from collections import Counter, OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report, roc_curve, auc)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR  = f"{DRIVE_DIR}/plantdoc"   # shared Drive folder
FIGURES_DIR = f"{OUTPUT_DIR}/figures"
MODELS_DIR  = f"{OUTPUT_DIR}/models"
TABLES_DIR  = f"{OUTPUT_DIR}/tables"
for d in [FIGURES_DIR, MODELS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------------
# FIX #8 -- the v6 student used 42,607 params = 41.6 KB INT8, i.e. 8% of the
# stated <500 KB budget. 96px throws away the fine texture that separates
# Tomato Septoria (many small spots) from Potato Early Blight (concentric rings).
# 128px + wider channels lands around 230 KB INT8 -- still comfortably inside budget,
# and ~250-400 ms/inference on an ESP32-S3 with PSRAM.
# ------------------------------------------------------------------
IMG_SIZE          = 128
TEACHER_IMG_SIZE  = 224
NUM_CLASSES       = 3
STUDENT_CHANNELS  = (32, 48, 64, 96, 96)     # v6 was (16, 24, 32, 48, 48)

BATCH_SIZE        = 48    # source
TARGET_BATCH_SIZE = 32    # FIX #7: smaller batch + drop_last=False keeps all 364 target images
TEACHER_BATCH_SIZE= 24

CLASSES = ["Tomato_Septoria_Leaf_Spot", "Potato_Early_Blight", "Corn_Leaf_Blight"]

# ---------------- stage switches ----------------
RUN_SANITY_GATE        = False    # Part 0B: source-only run. DO NOT SKIP on the first execution.
USE_BG_RANDOMIZATION   = True    # FIX #6
USE_TEACHER_KD         = True
TEACHER_USE_PSEUDO     = False   # FIX #1: OFF. This is what poisoned v6. Turn on only after
                                 #         the teacher's clean baseline is known.
STUDENT_USE_PSEUDO     = True    # gated + auto-disabling (see Part 3/4)
RUN_MULTI_SEED         = False    # Part 7 (expensive: re-runs the student per seed) -- turned ON:
                                  # a single-seed number is not defensible on a 79-image test set
SEEDS                  = [42, 1337, 2024]

# ---------------- teacher ----------------
TEACHER_BACKBONE   = 'mobilenet_v3_large'
TEACHER_EPOCHS     = 80
TEACHER_LR_BACKBONE= 3e-4        # FIX #10: discriminative LR instead of a cosmetic block freeze
TEACHER_LR_HEAD    = 3e-3
TEACHER_WARMUP     = 3
TEACHER_PATIENCE   = 30

# ---------------- student ----------------
EPOCHS         = 120
LR             = 1.5e-3
WEIGHT_DECAY   = 1e-4
LABEL_SMOOTHING= 0.1
WARMUP_EPOCHS  = 5
PATIENCE       = 40

# ---------------- domain adaptation ----------------
# EXPERIMENT: IM-only. v7 ran four alignment losses together (CORAL + MMD + IM + MCC).
# This run switches all four off except Information Maximization to see how much of the
# adaptation the plain IM term can carry on its own -- CORAL/MMD/MCC stay fully implemented
# below (Part 3) and can be flipped back on by setting these four switches to True.
USE_CORAL = False
USE_MMD   = False
USE_IM    = True
USE_MCC   = False

# FIX #9: CORAL down from 5.0. In v6 CORAL fell to 1e-4 while target accuracy never moved --
# the classic signature of degenerate alignment (domains matched by destroying class structure).
LAMBDA_CORAL_MAX  = 0.5
LAMBDA_MMD_MAX    = 0.5
LAMBDA_IM_MAX     = 0.5    # FIX #3: Information Maximization replaces bare entropy-min
LAMBDA_MCC_MAX    = 1.0    # FIX #3: Minimum Class Confusion -- class-aware, stable
LAMBDA_PSEUDO_MAX = 0.3

# FIX #2: v6 iterated the source loader (41 batches) and cycled the 5-batch target loader
# 8.2x per epoch, so 364 target images dominated every alignment term. Cap it.
DA_TARGET_PASSES  = 2

# ---------------- pseudo-labeling (FIX #1) ----------------
PSEUDO_CONF            = 0.95
PSEUDO_TOPK_PER_CLASS  = 8      # class-balanced instead of a global threshold
PSEUDO_MIN_MONITOR_ACC = 0.85   # hard safety valve: below this, self-training switches itself off
PSEUDO_PLATEAU_EPOCHS  = 8      # only start once target-val has stopped improving
EMA_DECAY              = 0.999

# ---------------- knowledge distillation (FIX #4) ----------------
LAMBDA_KD      = 1.0    # constant, NOT ramped, and no longer 3x the CE weight
LAMBDA_FEATKD  = 1.0
KD_TEMPERATURE = 4.0

SMOOTH_WINDOW = 5

CFG = dict(coral=LAMBDA_CORAL_MAX, mmd=LAMBDA_MMD_MAX, im=LAMBDA_IM_MAX, mcc=LAMBDA_MCC_MAX,
           pseudo=LAMBDA_PSEUDO_MAX, kd=LAMBDA_KD, featkd=LAMBDA_FEATKD, T=KD_TEMPERATURE,
           da_passes=DA_TARGET_PASSES, pseudo_conf=PSEUDO_CONF,
           pseudo_topk=PSEUDO_TOPK_PER_CLASS, pseudo_min_acc=PSEUDO_MIN_MONITOR_ACC)

print(f"\nIMG={IMG_SIZE} channels={STUDENT_CHANNELS} BS={BATCH_SIZE}/{TARGET_BATCH_SIZE}")
print(f"DA technique switches: CORAL={USE_CORAL} MMD={USE_MMD} IM={USE_IM} MCC={USE_MCC}")
print(f"DA lambdas (only active ones apply): coral={LAMBDA_CORAL_MAX} mmd={LAMBDA_MMD_MAX} "
      f"IM={LAMBDA_IM_MAX} MCC={LAMBDA_MCC_MAX}")
print(f"KD: lambda={LAMBDA_KD} (constant) T={KD_TEMPERATURE} feat={LAMBDA_FEATKD}")
print(f"Pseudo: teacher={TEACHER_USE_PSEUDO} student={STUDENT_USE_PSEUDO} "
      f"conf>={PSEUDO_CONF} topk={PSEUDO_TOPK_PER_CLASS}/class auto-off<{PSEUDO_MIN_MONITOR_ACC}")

Device: cuda | PyTorch: 2.11.0+cu130
GPU: Tesla T4

IMG=128 channels=(32, 48, 64, 96, 96) BS=48/32
DA technique switches: CORAL=False MMD=False IM=True MCC=False
DA lambdas (only active ones apply): coral=0.5 mmd=0.5 IM=0.5 MCC=1.0
KD: lambda=1.0 (constant) T=4.0 feat=1.0
Pseudo: teacher=False student=True conf>=0.95 topk=8/class auto-off<0.85


In [ ]:
# ============================================================
# PART 1A: DATASET INDEX
# ============================================================
SOURCE_ALL    = PV_COLOR
SEGMENTED_ALL = PV_SEG
TARGET_ALL    = PD_ROOT

SOURCE_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato___Septoria_leaf_spot"],
    "Potato_Early_Blight":       ["Potato___Early_blight"],
    "Corn_Leaf_Blight":          ["Corn_(maize)___Northern_Leaf_Blight"],
}
TARGET_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato Septoria leaf spot"],
    "Potato_Early_Blight":       ["Potato leaf early blight"],
    "Corn_Leaf_Blight":          ["Corn leaf blight"],
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class LeafIndex:
    """Path/label index only -- no transforms. Walks root_dir recursively and pools
    every image under any folder whose normalized name matches an alias for a class."""

    def __init__(self, root_dir, classes, class_dir_map, domain_label=0, name=""):
        self.img_paths, self.labels = [], []
        self.classes, self.domain_label = classes, domain_label

        def _norm(s):
            return s.strip().lower().replace(" ", "").replace("_", "")

        for idx, cls in enumerate(classes):
            alias = {_norm(a) for a in class_dir_map.get(cls, [cls])}
            matched = [dp for dp, _, _ in os.walk(root_dir) if _norm(os.path.basename(dp)) in alias]
            for d in matched:
                for f in sorted(os.listdir(d)):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        self.img_paths.append(os.path.join(d, f)); self.labels.append(idx)
            if not matched:
                print(f"    WARNING: no folder matched '{cls}' under {root_dir}")

        print(f"  [{name}] {len(self.img_paths)} images")
        for i, c in enumerate(classes):
            print(f"    {c}: {self.labels.count(i)}")

    def subset(self, indices):
        s = copy.copy(self)
        s.img_paths = [self.img_paths[i] for i in indices]
        s.labels    = [self.labels[i] for i in indices]
        return s


def stratified_split_indices(labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    idx_all = list(range(len(labels)))
    idx_tv, idx_test = train_test_split(idx_all, test_size=test_frac, stratify=labels,
                                        random_state=seed)
    lab_tv = [labels[i] for i in idx_tv]
    idx_tr, idx_val = train_test_split(idx_tv, test_size=val_frac / (1 - test_frac),
                                       stratify=lab_tv, random_state=seed)
    return idx_tr, idx_val, idx_test


for nm, p in [('PlantVillage color', SOURCE_ALL), ('PlantVillage segmented', SEGMENTED_ALL),
              ('PlantDoc', TARGET_ALL)]:
    print(f"  {nm:24s}: {'OK ' if os.path.isdir(p) else '!! NOT FOUND !!'} {p}")

print("\n--- Source (PlantVillage) ---")
src_index = LeafIndex(SOURCE_ALL, CLASSES, SOURCE_CLASS_DIRS, 0, "Source")
print("\n--- Target (PlantDoc) ---")
tgt_index = LeafIndex(TARGET_ALL, CLASSES, TARGET_CLASS_DIRS, 1, "Target")

src_tr_i, src_va_i, src_te_i = stratified_split_indices(src_index.labels)
tgt_tr_i, tgt_va_i, tgt_te_i = stratified_split_indices(tgt_index.labels)

src_train_idx = src_index.subset(src_tr_i)
src_val_idx   = src_index.subset(src_va_i)
src_test_idx  = src_index.subset(src_te_i)
tgt_train_idx = tgt_index.subset(tgt_tr_i)
tgt_val_idx   = tgt_index.subset(tgt_va_i)
tgt_test_idx  = tgt_index.subset(tgt_te_i)

print(f"\nSource  train/val/test: {len(src_train_idx.labels)}/"
      f"{len(src_val_idx.labels)}/{len(src_test_idx.labels)}")
print(f"Target  train/val/test: {len(tgt_train_idx.labels)}/"
      f"{len(tgt_val_idx.labels)}/{len(tgt_test_idx.labels)}")
print(f"\nNOTE: target test = {len(tgt_test_idx.labels)} images -> 1 image = "
      f"{100/len(tgt_test_idx.labels):.2f} accuracy points. Never quote a single point estimate.")

  PlantVillage color      : OK  /kaggle/input/plantvillage-dataset/plantvillage dataset/color
  PlantVillage segmented  : OK  /kaggle/input/plantvillage-dataset/plantvillage dataset/segmented
  PlantDoc                : OK  /kaggle/input/plantdoc-dataset

--- Source (PlantVillage) ---
  [Source] 3756 images
    Tomato_Septoria_Leaf_Spot: 1771
    Potato_Early_Blight: 1000
    Corn_Leaf_Blight: 985

--- Target (PlantDoc) ---
  [Target] 522 images
    Tomato_Septoria_Leaf_Spot: 157
    Potato_Early_Blight: 171
    Corn_Leaf_Blight: 194

Source  train/val/test: 2628/564/564
Target  train/val/test: 364/79/79

NOTE: target test = 79 images -> 1 image = 1.27 accuracy points. Never quote a single point estimate.


In [ ]:
# ============================================================
# PART 1B: BACKGROUND RANDOMIZATION  (FIX #6)
# ============================================================
# PlantVillage: centred leaf, uniform grey backdrop. PlantDoc: leaf in a cluttered field.
# A source-only model latches onto the backdrop as a shortcut. Cutting the leaf out and
# pasting it onto real field backgrounds removes that shortcut at the data level, which is
# more effective than any loss-side alignment term.
#
# Mask source, in order of preference:
#   1. PlantVillage's own segmented/ split (exact, free) -- matched by filename stem
#   2. Border-colour heuristic + morphological cleanup (works because the backdrop is uniform)
# Backgrounds: random crops of PlantDoc TRAIN images only (never val/test -> no leakage).

from scipy import ndimage as ndi


def build_segmented_lookup(seg_root, class_dir_map, classes):
    """stem -> segmented path. PlantVillage segmented filenames carry a '_final_masked'
    suffix, so match on the leading UUID-ish stem rather than the full filename."""
    lut = {}
    if not (seg_root and os.path.isdir(seg_root)):
        return lut

    def _norm(s):
        return s.strip().lower().replace(" ", "").replace("_", "")

    alias = set()
    for c in classes:
        alias |= {_norm(a) for a in class_dir_map.get(c, [c])}
    for dp, _, files in os.walk(seg_root):
        if _norm(os.path.basename(dp)) not in alias:
            continue
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                stem = os.path.splitext(f)[0].replace('_final_masked', '')
                lut[stem] = os.path.join(dp, f)
    return lut


SEG_LUT = build_segmented_lookup(SEGMENTED_ALL, SOURCE_CLASS_DIRS, CLASSES) if USE_BG_RANDOMIZATION else {}
print(f"Segmented-mask lookup: {len(SEG_LUT)} entries "
      f"({'exact masks available' if SEG_LUT else 'falling back to the border heuristic'})")


def estimate_leaf_mask(img, seg_path=None):
    """Boolean HxW mask, True = leaf. Returns None when the result looks untrustworthy,
    in which case the caller simply skips background randomization for that image."""
    if seg_path is not None and os.path.exists(seg_path):
        try:
            seg = np.asarray(Image.open(seg_path).convert("RGB").resize(img.size), np.float32)
            m = seg.sum(2) > 24          # segmented split paints the background pure black
            if 0.05 < m.mean() < 0.97:
                return ndi.binary_closing(m, np.ones((5, 5)))
        except Exception:
            pass

    a = np.asarray(img.convert("RGB"), np.float32)
    b = np.concatenate([a[:6].reshape(-1, 3), a[-6:].reshape(-1, 3),
                        a[:, :6].reshape(-1, 3), a[:, -6:].reshape(-1, 3)])
    med = np.median(b, 0)
    d = np.linalg.norm(a - med, axis=2)
    m = d > max(35.0, np.percentile(d, 40))
    m = ndi.binary_opening(m, np.ones((3, 3)))
    m = ndi.binary_closing(m, np.ones((9, 9)))
    m = ndi.binary_fill_holes(m)

    lab, n = ndi.label(m)                # keep only the largest blob (the leaf)
    if n > 1:
        sizes = ndi.sum(m, lab, range(1, n + 1))
        m = lab == (int(np.argmax(sizes)) + 1)
    if not (0.05 < m.mean() < 0.95):
        return None
    return m


def build_background_bank(index_obj, n=180, size=320):
    """Field backgrounds taken from PlantDoc TRAIN images only."""
    rng = np.random.RandomState(SEED)
    paths = list(index_obj.img_paths)
    rng.shuffle(paths)
    bank = []
    for p in paths[:n]:
        try:
            bank.append(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))
        except Exception:
            continue
    return bank


BG_BANK = build_background_bank(tgt_train_idx) if USE_BG_RANDOMIZATION else []
print(f"Background bank: {len(BG_BANK)} field images (PlantDoc TRAIN split only)")


MASK_SIZE = 192


def precompute_masks(index_obj):
    """Masks are computed ONCE here and stored bit-packed (~6 KB each), instead of being
    recomputed inside the dataloader on every __getitem__. Without this the scipy
    morphology becomes the training bottleneck."""
    if not USE_BG_RANDOMIZATION:
        return {}
    cache, t0 = {}, time.time()
    for n, p in enumerate(index_obj.img_paths):
        try:
            im = Image.open(p).convert("RGB").resize((MASK_SIZE, MASK_SIZE), Image.BILINEAR)
            stem = os.path.splitext(os.path.basename(p))[0]
            m = estimate_leaf_mask(im, SEG_LUT.get(stem))
            if m is not None:
                cache[p] = np.packbits(m)
        except Exception:
            pass
        if (n + 1) % 500 == 0:
            print(f"    masks {n+1}/{len(index_obj.img_paths)} ({time.time()-t0:.0f}s)")
    print(f"  usable leaf masks: {len(cache)}/{len(index_obj.img_paths)} "
          f"in {time.time()-t0:.0f}s")
    return cache


MASK_CACHE = precompute_masks(src_train_idx) if USE_BG_RANDOMIZATION else {}


def randomize_background(img, path, rng):
    """Composite the leaf onto a random field background with a soft alpha edge."""
    if not BG_BANK or path not in MASK_CACHE:
        return img
    m = np.unpackbits(MASK_CACHE[path])[:MASK_SIZE * MASK_SIZE].reshape(MASK_SIZE, MASK_SIZE)

    bg = BG_BANK[rng.randint(len(BG_BANK))]
    W, H = img.size
    bw, bh = bg.size
    cw = rng.randint(int(bw * 0.4), bw + 1)
    x0 = rng.randint(0, bw - cw + 1); y0 = rng.randint(0, bh - cw + 1)
    bg = bg.crop((x0, y0, x0 + cw, y0 + cw)).resize((W, H), Image.BILINEAR)

    alpha = (Image.fromarray((m * 255).astype(np.uint8))
             .resize((W, H), Image.BILINEAR).filter(ImageFilter.GaussianBlur(1.5)))
    out = Image.composite(img, bg, alpha)
    if rng.rand() < 0.3:                 # sometimes blur the background only
        out = Image.composite(out, bg.filter(ImageFilter.GaussianBlur(2.0)), alpha)
    return out

Segmented-mask lookup: 3756 entries (exact masks available)
Background bank: 180 field images (PlantDoc TRAIN split only)
    masks 500/2628 (12s)
    masks 1000/2628 (24s)
    masks 1500/2628 (36s)
    masks 2000/2628 (48s)
    masks 2500/2628 (60s)
  usable leaf masks: 2628/2628 in 63s


In [ ]:
# ============================================================
# PART 1C: DualResDataset  (FIX #4 + FIX #6)
# ============================================================
# One augmentation, rendered at both resolutions. This is the cell that repairs distillation:
# v6 augmented at 96px and bilinearly upsampled to 224px for the teacher, so the teacher saw
# blurry stretched inputs it was never fine-tuned on and emitted soft targets that were then
# weighted 3x the CE loss. Here both views come from the same 256px augmented image.
#
# Returns (x_student, x_teacher, x_weak_student, x_weak_teacher, label, domain).
# Two weak views are needed because pseudo-labels must be generated at whatever resolution the
# model being trained consumes -- 128px for the student, 224px for the teacher.
# Unrequested views are 1-element placeholders so the default collate still stacks cleanly.

_PLACEHOLDER = torch.zeros(1)


class DualResDataset(Dataset):
    def __init__(self, index_obj, train=True, emit_teacher=False, emit_weak=False,
                 bg_random_p=0.0, img_size=IMG_SIZE, teacher_size=TEACHER_IMG_SIZE):
        self.idx = index_obj
        self.train = train
        self.emit_teacher = emit_teacher
        self.emit_weak = emit_weak
        self.bg_p = bg_random_p if index_obj.domain_label == 0 else 0.0
        self.img_size, self.teacher_size = img_size, teacher_size

        # -- strong, domain-gap-aware augmentation (FIX #6) --
        # RandomResizedCrop(0.25-1.0) is the important one: it forces the model onto lesion
        # texture instead of whole-leaf silhouette, which does not survive the domain shift.
        self.geo_train = transforms.Compose([
            transforms.RandomResizedCrop(256, scale=(0.25, 1.0), ratio=(0.75, 1.33)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.2),
            transforms.RandomApply([transforms.RandomRotation(25)], p=0.5),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.1),
            transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 2.0))], p=0.3),
        ])
        self.geo_eval = transforms.Resize((256, 256))
        self.geo_weak = transforms.Compose([transforms.Resize((256, 256)),
                                            transforms.RandomHorizontalFlip(0.5)])

        self.to_student = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)])
        self.to_teacher = transforms.Compose([
            transforms.Resize((teacher_size, teacher_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        self.erase = transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), value='random')

    def __len__(self):
        return len(self.idx.img_paths)

    def __getitem__(self, i):
        path = self.idx.img_paths[i]
        img = Image.open(path).convert("RGB")
        y, d = self.idx.labels[i], self.idx.domain_label

        if self.train and self.bg_p > 0:
            # seeded from the worker's torch RNG (was wall-clock time -> not reproducible)
            rng = np.random.RandomState(int(torch.randint(0, 2 ** 31 - 1, (1,)).item()))
            if rng.rand() < self.bg_p:
                img = randomize_background(img.resize((256, 256), Image.BILINEAR), path, rng)

        base = self.geo_train(img) if self.train else self.geo_eval(img)

        xs = self.to_student(base)
        if self.train:
            xs = self.erase(xs)
        xt = self.to_teacher(base) if self.emit_teacher else _PLACEHOLDER
        if self.emit_weak:
            wbase = self.geo_weak(img)
            xw_s = self.to_student(wbase)
            xw_t = self.to_teacher(wbase) if self.emit_teacher else _PLACEHOLDER
        else:
            xw_s = xw_t = _PLACEHOLDER
        return xs, xt, xw_s, xw_t, y, d

In [ ]:
# ============================================================
# PART 2: MicroBrainNet-DA v7
# ============================================================

class InvertedResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, expand_ratio=2):
        super().__init__()
        self.use_residual = (stride == 1 and in_ch == out_ch)
        hid = in_ch * expand_ratio
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, hid, 1, bias=False), nn.BatchNorm2d(hid),
                       nn.ReLU6(inplace=True)]
        layers += [nn.Conv2d(hid, hid, 3, stride, 1, groups=hid, bias=False),
                   nn.BatchNorm2d(hid), nn.ReLU6(inplace=True),
                   nn.Conv2d(hid, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.conv(x) if self.use_residual else self.conv(x)


class SqueezeExciteBlock(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__()
        mid = max(ch // reduction, 8)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(nn.Linear(ch, mid), nn.ReLU(inplace=True),
                                    nn.Linear(mid, ch), nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.excite(self.squeeze(x).view(b, c)).view(b, c, 1, 1)


class MicroViTBlock(nn.Module):
    """Linear attention, O(n). ReLU6 in the FFN (not GELU) so every op has a TFLite Micro kernel."""

    def __init__(self, dim, num_tokens, ffn_ratio=2):
        super().__init__()
        self.q_proj = nn.Linear(dim, dim); self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim); self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim); self.norm2 = nn.LayerNorm(dim)
        # learned positional embedding -- v6 had none, so patch layout was invisible
        self.pos = nn.Parameter(torch.zeros(1, num_tokens, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        h = dim * ffn_ratio
        self.ffn = nn.Sequential(nn.Linear(dim, h), nn.ReLU6(inplace=True), nn.Dropout(0.1),
                                 nn.Linear(h, dim), nn.Dropout(0.1))

    def linear_attention(self, q, k, v):
        q = F.elu(q) + 1
        k = F.elu(k) + 1
        kv = torch.bmm(k.transpose(1, 2), v)
        num = torch.bmm(q, kv)
        den = torch.bmm(q, k.sum(1, keepdim=True).transpose(1, 2)).clamp(min=1e-4)
        return num / den

    def forward(self, x):
        x = x + self.pos
        xn = self.norm1(x)
        x = x + self.out_proj(self.linear_attention(self.q_proj(xn), self.k_proj(xn),
                                                    self.v_proj(xn)))
        return x + self.ffn(self.norm2(x))


class MicroBrainNetDA(nn.Module):
    """128x128x3 RGB -> 3 logits. Downsampling: stem s2, b1 s2, b2 s2, b3 s2, b4 s1
    => 8x8 = 64 tokens for the ViT stage."""

    def __init__(self, num_classes=NUM_CLASSES, channels=STUDENT_CHANNELS, img_size=IMG_SIZE,
                 use_se=True, use_vit=True):
        super().__init__()
        c0, c1, c2, c3, c4 = channels
        self.use_se, self.use_vit = use_se, use_vit
        self.stem = nn.Sequential(nn.Conv2d(3, c0, 3, 2, 1, bias=False),
                                  nn.BatchNorm2d(c0), nn.ReLU6(inplace=True))
        self.block1 = InvertedResidualBlock(c0, c1, 2, 2)
        self.block2 = InvertedResidualBlock(c1, c2, 2, 2)
        self.block3 = InvertedResidualBlock(c2, c3, 2, 2)
        self.block4 = InvertedResidualBlock(c3, c4, 1, 2)
        self.se = SqueezeExciteBlock(c4) if use_se else nn.Identity()
        self.grid = img_size // 16
        self.vit = MicroViTBlock(c4, self.grid * self.grid) if use_vit else nn.Identity()
        self.classifier = nn.Sequential(nn.LayerNorm(c4), nn.Dropout(0.2),
                                        nn.Linear(c4, num_classes))
        self.feature_dim = c4
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def extract_features(self, x):
        x = self.stem(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.se(x)
        if self.use_vit:
            x = x.flatten(2).transpose(1, 2)
            x = self.vit(x)
            return x.mean(1)
        return F.adaptive_avg_pool2d(x, 1).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


class TeacherNet(nn.Module):
    """ImageNet-pretrained backbone behind the same extract_features()/classifier interface,
    so it drops straight into train_da_epoch(). Training-time only -- never quantized."""

    def __init__(self, num_classes=NUM_CLASSES, backbone_name=TEACHER_BACKBONE):
        super().__init__()
        if backbone_name == 'mobilenet_v3_large':
            bb = tvm.mobilenet_v3_large(weights=tvm.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        elif backbone_name == 'mobilenet_v3_small':
            bb = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        elif backbone_name == 'efficientnet_b0':
            bb = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        else:
            raise ValueError(backbone_name)
        self.features = bb.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feature_dim = bb.classifier[0].in_features
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.feature_dim, num_classes))

    def extract_features(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


model = MicroBrainNetDA().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Student params : {total_params:,}")
print(f"  FP32 : {total_params*4/1024:.1f} KB")
print(f"  INT8 : {total_params/1024:.1f} KB   (budget 500 KB -> "
      f"{total_params/1024/500*100:.0f}% used; v6 used 8%)")
print(f"  ViT token grid: {model.grid}x{model.grid} = {model.grid**2} tokens")
with torch.no_grad():
    print("forward:", tuple(model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)).shape))

Student params : 167,131
  FP32 : 652.9 KB
  INT8 : 163.2 KB   (budget 500 KB -> 33% used; v6 used 8%)
  ViT token grid: 8x8 = 64 tokens
forward: (2, 3)


In [ ]:
# ============================================================
# PART 3: LOSS FUNCTIONS
# ============================================================

def standardize_feats(x, eps=1e-5):
    return (x - x.mean(0, keepdim=True)) / (x.std(0, keepdim=True) + eps)


def compute_covariance(x):
    n = x.size(0)
    ones = torch.ones(1, n, device=x.device)
    mean = ones @ x / n
    return (x.t() @ x - mean.t() @ mean) / max(n - 1, 1)


def coral_loss(fs, ft):
    s, t = standardize_feats(fs), standardize_feats(ft)
    d = s.size(1)
    return (compute_covariance(s) - compute_covariance(t)).pow(2).sum() / (4.0 * d * d)


def gaussian_kernel_matrix(s, t, kernel_mul=2.0, kernel_num=5):
    total = torch.cat([s, t], 0)
    n = total.size(0)
    l2 = ((total.unsqueeze(0) - total.unsqueeze(1)) ** 2).sum(2)
    bw = torch.sum(l2.detach()) / max(n * n - n, 1)
    bw = bw / (kernel_mul ** (kernel_num // 2)) + 1e-8
    return sum(torch.exp(-l2 / (bw * kernel_mul ** i)) for i in range(kernel_num))


def mmd_loss(fs, ft):
    s, t = standardize_feats(fs), standardize_feats(ft)
    ns = s.size(0)
    k = gaussian_kernel_matrix(s, t)
    return k[:ns, :ns].mean() + k[ns:, ns:].mean() - k[:ns, ns:].mean() - k[ns:, :ns].mean()


def im_loss(logits):
    """Information Maximization = confident per sample MINUS diverse across the batch.

    FIX #3. Bare entropy minimization (v6) has a degenerate optimum: predict one class for
    everything. The batch-marginal term makes that the WORST solution instead of the best.
    Returns ent_individual - ent_batch; both terms are minimized together."""
    p = F.softmax(logits, 1)
    ent_ind = -(p * torch.log(p + 1e-8)).sum(1).mean()
    pm = p.mean(0)
    ent_batch = -(pm * torch.log(pm + 1e-8)).sum()
    return ent_ind - ent_batch


def mcc_loss(logits, T=2.5):
    """Minimum Class Confusion (Jin et al.). Class-aware, so unlike CORAL/MMD it cannot be
    satisfied by a feature collapse -- off-diagonal mass only falls if target predictions
    become mutually distinct."""
    p = F.softmax(logits / T, 1)
    w = (1 + (-(p * torch.log(p + 1e-8)).sum(1))).detach()
    w = (w / w.sum()) * p.size(0)
    C = (p * w.unsqueeze(1)).t() @ p
    C = C / (C.sum(1, keepdim=True) + 1e-8)
    return (C.sum() - C.trace()) / logits.size(1)


def kd_loss_fn(s_logits, t_logits, T=4.0):
    return F.kl_div(F.log_softmax(s_logits / T, 1), F.softmax(t_logits / T, 1),
                    reduction='batchmean') * (T * T)


def feat_kd_loss(s_proj, t_feat):
    """Cosine feature distillation. With a ~70x capacity gap, logit-only KD transfers little;
    matching normalized feature direction gives the student a much denser signal."""
    return F.mse_loss(F.normalize(s_proj, dim=1), F.normalize(t_feat.detach(), dim=1))


@torch.no_grad()
def balanced_pseudo(logits, k_per_class=PSEUDO_TOPK_PER_CLASS, min_conf=PSEUDO_CONF,
                    n_classes=NUM_CLASSES):
    """Class-balanced top-k selection (FIX #1). A global threshold lets the dominant class
    supply nearly every pseudo-label, which is how v6 reinforced its own Potato->Tomato error."""
    probs = F.softmax(logits, 1)
    conf, lab = probs.max(1)
    keep = torch.zeros_like(conf, dtype=torch.bool)
    for c in range(n_classes):
        idx = ((lab == c) & (conf >= min_conf)).nonzero(as_tuple=True)[0]
        if idx.numel():
            top = conf[idx].topk(min(k_per_class, idx.numel())).indices
            keep[idx[top]] = True
    return keep, lab


def ramp(epoch, max_epoch, warm=0):
    """Sigmoid ramp for the alignment weights. KD is deliberately NOT ramped (FIX #4):
    ramping it up meant it peaked exactly when it should have been fading out."""
    if epoch <= warm:
        return 0.0
    p = (epoch - warm) / max(max_epoch - warm, 1)
    return float(2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)


print("Losses ready: coral, mmd, im (IM replaces entropy-min), mcc, kd, feat_kd, balanced_pseudo")

Losses ready: coral, mmd, im (IM replaces entropy-min), mcc, kd, feat_kd, balanced_pseudo


In [ ]:
# ============================================================
# PART 4: TRAINING UTILITIES
# ============================================================

class EMA:
    """Exponential moving average of the student, used only to generate pseudo-labels."""

    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-3):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best, self.early_stop, self.best_model = 0, None, False, None

    def __call__(self, score, model):
        if self.best is None or score > self.best + self.min_delta:
            self.best, self.counter = score, 0
            self.best_model = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


@torch.no_grad()
def evaluate(model, loader, criterion, device, which='s'):
    """which='s' -> 128px/[-1,1] student view; which='t' -> 224px/ImageNet teacher view."""
    model.eval()
    tot, preds, labs, probs = 0.0, [], [], []
    for xs, xt, xw_s, xw_t, y, d in loader:
        x = (xs if which == 's' else xt).to(device)
        y = y.to(device)
        logits = model(x)
        tot += criterion(logits, y).item() * x.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
        probs.extend(F.softmax(logits, 1).cpu().numpy())
    n = len(labs)
    return (tot / n, accuracy_score(labs, preds), f1_score(labs, preds, average='weighted'),
            np.array(preds), np.array(labs), np.array(probs))


@torch.no_grad()
def adapt_bn(model, loader, device, which='s', n_pass=3):
    """AdaBN (FIX #5): recompute BN running statistics from TARGET data only.
    momentum=None makes each BN accumulate a cumulative average over the passes.
    Must run before INT8 calibration, otherwise quantization bakes in the wrong statistics."""
    has_bn = False
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.reset_running_stats(); m.momentum = None; has_bn = True
    if not has_bn:
        return model
    model.train()
    for _ in range(n_pass):
        for xs, xt, xw_s, xw_t, y, d in loader:
            model((xs if which == 's' else xt).to(device))
    model.eval()
    return model


def train_da_epoch(model, src_loader, tgt_loader, criterion, optimizer, device,
                   epoch, max_epoch, cfg, which='s',
                   teacher=None, kd_proj=None, ema=None,
                   pseudo_enabled=False,
                   use_coral=True, use_mmd=True, use_im=True, use_mcc=True, use_kd=True,
                   kd_on_target=True):
    """One epoch of domain-adaptive (+ optionally distilled) training.

    Key structural differences from v6:
      * source and target go through ONE concatenated forward -> clean BN statistics (FIX #5)
      * alignment losses run on at most cfg['da_passes'] * len(tgt_loader) steps (FIX #2)
      * KD uses the paired native-resolution teacher view from DualResDataset (FIX #4)
      * pseudo-labels come from the EMA model on the WEAK view, applied to the STRONG view
    """
    model.train()
    lam = ramp(epoch, max_epoch)
    w = dict(coral=cfg['coral'] * lam if use_coral else 0.0,
             mmd=cfg['mmd'] * lam if use_mmd else 0.0,
             im=cfg['im'] * lam if use_im else 0.0,
             mcc=cfg['mcc'] * lam if use_mcc else 0.0,
             pseudo=cfg['pseudo'] * lam if pseudo_enabled else 0.0,
             kd=cfg['kd'] if (use_kd and teacher is not None) else 0.0,
             featkd=cfg['featkd'] if (use_kd and teacher is not None and kd_proj is not None) else 0.0)

    n_da = min(len(src_loader), max(1, cfg['da_passes'] * len(tgt_loader)))
    tgt_iter = iter(tgt_loader)

    acc_loss = {k: 0.0 for k in ['cls', 'coral', 'mmd', 'im', 'mcc', 'pseudo', 'kd', 'featkd']}
    n_seen, n_da_seen = 0, 0
    preds, labs = [], []
    ps_used, ps_correct = 0, 0

    for step, (xs_s, xs_t, _w1, _w2, ys, _d) in enumerate(src_loader):
        do_da = step < n_da
        x_src = (xs_s if which == 's' else xs_t).to(device, non_blocking=True)
        ys = ys.to(device, non_blocking=True)
        bs = x_src.size(0)

        if do_da:
            try:
                xt_s, xt_t, xt_ws, xt_wt, yt_true, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                xt_s, xt_t, xt_ws, xt_wt, yt_true, _ = next(tgt_iter)
            x_tgt = (xt_s if which == 's' else xt_t).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ---- single concatenated forward: clean BN statistics (FIX #5) ----
        if do_da:
            feats = model.extract_features(torch.cat([x_src, x_tgt], 0))
            f_s, f_t = feats[:bs], feats[bs:]
            logits_s, logits_t = model.classifier(f_s), model.classifier(f_t)
        else:
            f_s = model.extract_features(x_src)
            f_t, logits_t = None, None
            logits_s = model.classifier(f_s)

        L_cls = criterion(logits_s, ys)
        loss = L_cls
        parts = {k: 0.0 for k in acc_loss}
        parts['cls'] = L_cls.item()

        if do_da:
            if w['coral'] > 0:
                l = coral_loss(f_s, f_t);  loss = loss + w['coral'] * l;  parts['coral'] = l.item()
            if w['mmd'] > 0:
                l = mmd_loss(f_s, f_t);    loss = loss + w['mmd'] * l;    parts['mmd'] = l.item()
            if w['im'] > 0:
                l = im_loss(logits_t);     loss = loss + w['im'] * l;     parts['im'] = l.item()
            if w['mcc'] > 0:
                l = mcc_loss(logits_t);    loss = loss + w['mcc'] * l;    parts['mcc'] = l.item()

            if w['pseudo'] > 0 and ema is not None:
                with torch.no_grad():
                    # weak view at the resolution THIS model consumes -> pseudo-labels
                    xw = (xt_ws if which == 's' else xt_wt).to(device)
                    weak_logits = ema.model(xw)
                keep, plab = balanced_pseudo(weak_logits, cfg['pseudo_topk'], cfg['pseudo_conf'])
                if keep.sum() > 0:
                    l = F.cross_entropy(logits_t[keep], plab[keep])  # strong view -> trained on
                    loss = loss + w['pseudo'] * l
                    parts['pseudo'] = l.item()
                    ps_used += int(keep.sum())
                    ps_correct += int((plab[keep].cpu() == yt_true[keep.cpu()]).sum())

        # ---- distillation on the paired native-resolution teacher view (FIX #4) ----
        if w['kd'] > 0:
            with torch.no_grad():
                tf_s = teacher.extract_features(xs_t.to(device, non_blocking=True))
                tl_s = teacher.classifier(tf_s)
                if do_da and kd_on_target:
                    tf_t = teacher.extract_features(xt_t.to(device, non_blocking=True))
                    tl_t = teacher.classifier(tf_t)
            l = kd_loss_fn(logits_s, tl_s, cfg['T'])
            if do_da and kd_on_target:
                l = 0.5 * (l + kd_loss_fn(logits_t, tl_t, cfg['T']))
            loss = loss + w['kd'] * l
            parts['kd'] = l.item()

            if w['featkd'] > 0:
                lf = feat_kd_loss(kd_proj(f_s), tf_s)
                if do_da and kd_on_target:
                    lf = 0.5 * (lf + feat_kd_loss(kd_proj(f_t), tf_t))
                loss = loss + w['featkd'] * lf
                parts['featkd'] = lf.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if ema is not None:
            ema.update(model)

        for k in acc_loss:
            acc_loss[k] += parts[k] * bs
        n_seen += bs
        n_da_seen += bs if do_da else 0
        preds.extend(logits_s.argmax(1).detach().cpu().numpy())
        labs.extend(ys.cpu().numpy())

    out = {f'{k}_loss': v / max(n_seen, 1) for k, v in acc_loss.items()}
    out.update(train_acc=accuracy_score(labs, preds),
               train_f1=f1_score(labs, preds, average='weighted'),
               lam=lam, pseudo_used=ps_used,
               pseudo_acc=(ps_correct / ps_used) if ps_used else float('nan'),
               weights=w)
    return out


def make_scheduler(opt, epochs, warmup):
    def fn(e):
        return (e + 1) / max(warmup, 1) if e < warmup else \
               0.5 * (1 + math.cos(math.pi * (e - warmup) / max(epochs - warmup, 1)))
    return optim.lr_scheduler.LambdaLR(opt, fn)


cls_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print("Utilities ready: EMA, EarlyStopping, evaluate, adapt_bn, train_da_epoch, make_scheduler")

Utilities ready: EMA, EarlyStopping, evaluate, adapt_bn, train_da_epoch, make_scheduler


In [ ]:
# ============================================================
# LOADERS: eval loaders (deterministic) + per-run training loaders
# ============================================================
# Training loaders are rebuilt for every run with a seeded torch.Generator, so a run
# depends only on its own seed (shuffling, augmentation, background choice), never on
# what ran before it in the same session.
NW = 2          # Colab has 2 CPU cores (keep the same in every notebook of the set)

def mk_loader(ds, bs, shuffle, drop_last=False):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NW,
                      pin_memory=True, drop_last=drop_last, persistent_workers=(NW > 0))

BG_P = 0.5 if USE_BG_RANDOMIZATION else 0.0

src_train_ds      = DualResDataset(src_train_idx, True, emit_teacher=True, bg_random_p=BG_P)
src_train_ds_nobg = DualResDataset(src_train_idx, True, emit_teacher=True, bg_random_p=0.0)
tgt_train_ds      = DualResDataset(tgt_train_idx, True, emit_teacher=True, emit_weak=True)

src_val_ds  = DualResDataset(src_val_idx,  False, emit_teacher=True)
src_test_ds = DualResDataset(src_test_idx, False, emit_teacher=True)
tgt_val_ds  = DualResDataset(tgt_val_idx,  False, emit_teacher=True)
tgt_test_ds = DualResDataset(tgt_test_idx, False, emit_teacher=True)

src_val_loader  = mk_loader(src_val_ds,  BATCH_SIZE, False)
src_test_loader = mk_loader(src_test_ds, BATCH_SIZE, False)
tgt_val_loader  = mk_loader(tgt_val_ds,  BATCH_SIZE, False)
tgt_test_loader = mk_loader(tgt_test_ds, BATCH_SIZE, False)


def make_train_loaders(seed, bg=True):
    g_src = torch.Generator(); g_src.manual_seed(int(seed))
    g_tgt = torch.Generator(); g_tgt.manual_seed(int(seed) + 7)
    src = DataLoader(src_train_ds if bg else src_train_ds_nobg, batch_size=BATCH_SIZE,
                     shuffle=True, num_workers=NW, pin_memory=True, drop_last=True,
                     persistent_workers=(NW > 0), generator=g_src)
    tgt = DataLoader(tgt_train_ds, batch_size=TARGET_BATCH_SIZE, shuffle=True,
                     num_workers=NW, pin_memory=True, drop_last=False,
                     persistent_workers=(NW > 0), generator=g_tgt)
    return src, tgt


_s, _t = make_train_loaders(SEED)
print(f"source batches/epoch: {len(_s)} | target batches/epoch: {len(_t)} | "
      f"DA steps/epoch: {min(len(_s), DA_TARGET_PASSES * len(_t))}")
print(f"target test images: {len(tgt_test_ds)} (1 image = {100/len(tgt_test_ds):.2f} points)")
del _s, _t

source batches/epoch: 54 | target batches/epoch: 12 | DA steps/epoch: 24
target test images: 79 (1 image = 1.27 points)


In [ ]:
# ============================================================
# RESULT STORE: every finished run is written to disk immediately
# ============================================================
# Runs already present in runs.csv are skipped, so a session that stops half-way can be
# continued in a new session. (On Colab everything is on Drive, so this is automatic.)
#   1. "Save Version" (Save & Run All or Quick Save with outputs) at the end of a session.
#   2. In the next session, add that version's output as an input dataset.
#   3. Put its results folder path in PREV_RESULTS_DIRS below.
import shutil, hashlib

RESULTS_DIR = f"{OUTPUT_DIR}/results"
PRED_DIR    = f"{RESULTS_DIR}/preds"
RUNS_CSV    = f"{RESULTS_DIR}/runs_{NB_TAG}.csv"   # one file per notebook
os.makedirs(PRED_DIR, exist_ok=True)

PREV_RESULTS_DIRS = [
    # e.g. "/kaggle/input/tlf-ablation-session1/tlf_runs/results",
]


def restore_previous(dirs):
    frames = []
    if os.path.exists(RUNS_CSV):
        frames.append(pd.read_csv(RUNS_CSV))
    for d in dirs:
        if not os.path.isdir(d):
            print(f"  !! not found, skipped: {d}")
            continue
        if os.path.exists(f"{d}/runs.csv"):
            frames.append(pd.read_csv(f"{d}/runs.csv"))
        for f in glob.glob(f"{d}/preds/*.npz"):
            dst = f"{PRED_DIR}/{os.path.basename(f)}"
            if not os.path.exists(dst):
                shutil.copy(f, dst)
        for f in glob.glob(f"{d}/*.pth"):
            dst = f"{RESULTS_DIR}/{os.path.basename(f)}"
            if not os.path.exists(dst):
                shutil.copy(f, dst)
        print(f"  restored from {d}")
    if frames:
        df = pd.concat(frames, ignore_index=True)
        df = df.drop_duplicates(subset=['name', 'seed'], keep='last')
        df.to_csv(RUNS_CSV, index=False)
        print(f"  {len(df)} finished runs on record")


def load_runs():
    """All runs of the set: every runs_*.csv in the shared results folder."""
    files = sorted(glob.glob(f"{RESULTS_DIR}/runs*.csv"))
    if not files:
        return pd.DataFrame()
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return df.drop_duplicates(subset=['name', 'seed'], keep='last').reset_index(drop=True)


def is_done(name, seed):
    df = load_runs()
    return (not df.empty) and bool(((df['name'] == name) & (df['seed'] == seed)).any())


def append_run(row):
    df = pd.read_csv(RUNS_CSV) if os.path.exists(RUNS_CSV) else pd.DataFrame()
    if not df.empty:
        df = df[~((df['name'] == row['name']) & (df['seed'] == row['seed']))]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(RUNS_CSV, index=False)


def pred_path(name, seed):
    safe = "".join(ch if ch.isalnum() else "_" for ch in name)
    return f"{PRED_DIR}/{safe}__s{seed}.npz"


def file_md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()[:12]


SESSION_START     = time.time()
MAX_SESSION_HOURS = 11.0          # Colab may stop earlier; finished runs are on Drive anyway


def time_left_for(est_hours):
    used = (time.time() - SESSION_START) / 3600
    return used + est_hours < MAX_SESSION_HOURS


def estimated_run_hours(default=1.1):
    df = load_runs()
    if df.empty or 'time_s' not in df:
        return default
    return float(df['time_s'].mean()) / 3600 * 1.1


restore_previous(PREV_RESULTS_DIRS)

In [ ]:
# ============================================================
# TRAINING FUNCTIONS: teacher and student, same protocol as the main notebook
# ============================================================
# Differences from the main notebook, all deliberate:
#   * Pseudo-label auto-off is kept exactly as in the main notebook (PSEUDO_AUTO_OFF=True):
#     pseudo-labeling is switched off when its accuracy, measured with TARGET-TRAIN labels,
#     falls below PSEUDO_MIN_MONITOR_ACC. Each run records whether it ever fired
#     (auto_off_fired), so the paper can report it.
#   * Training loaders are rebuilt per run with a seeded generator (reproducibility).
#   * Every variant uses the full budget (EPOCHS, PATIENCE) and the same model selection,
#     so all rows of a table are directly comparable.

PSEUDO_AUTO_OFF = True   # same as the main notebook


def bootstrap_ci(labels, preds, n_boot=5000, seed=SEED):
    rng = np.random.RandomState(seed)
    labels, preds = np.asarray(labels), np.asarray(preds)
    n = len(labels)
    accs = np.array([(labels[i] == preds[i]).mean()
                     for i in (rng.randint(0, n, n) for _ in range(n_boot))])
    return accs.mean(), *np.percentile(accs, [2.5, 97.5])


def _freeze(m):
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m


def train_teacher(use_im=True, seed=SEED, tag="teacher"):
    """Same recipe as Part 5 of the main notebook (pseudo-labeling off for the teacher)."""
    set_seed(seed)
    src_l, tgt_l = make_train_loaders(seed, bg=True)
    t = TeacherNet().to(DEVICE)
    head_ids = {id(p) for p in t.classifier.parameters()}
    opt = optim.AdamW([
        {'params': [p for p in t.parameters() if id(p) not in head_ids], 'lr': TEACHER_LR_BACKBONE},
        {'params': list(t.classifier.parameters()), 'lr': TEACHER_LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)
    sched = make_scheduler(opt, TEACHER_EPOCHS, TEACHER_WARMUP)
    stop = EarlyStopping(patience=TEACHER_PATIENCE, min_delta=1e-3)
    raw, best_sm, best_state, t0 = [], 0.0, copy.deepcopy(t.state_dict()), time.time()
    print(f"[{tag}] training | IM={'on' if use_im else 'off'} | {TEACHER_EPOCHS} epochs")
    for ep in range(1, TEACHER_EPOCHS + 1):
        st = train_da_epoch(t, src_l, tgt_l, cls_criterion, opt, DEVICE, ep, TEACHER_EPOCHS,
                            CFG, which='t', teacher=None, kd_proj=None, ema=None,
                            pseudo_enabled=False, use_kd=False, use_coral=False,
                            use_mmd=False, use_im=use_im, use_mcc=False)
        sched.step()
        _, tv, _, _, _, _ = evaluate(t, tgt_val_loader, cls_criterion, DEVICE, 't')
        raw.append(tv)
        sm = float(np.mean(raw[-SMOOTH_WINDOW:]))
        if sm > best_sm:
            best_sm, best_state = sm, copy.deepcopy(t.state_dict())
        stop(sm, t)
        if ep % 10 == 0 or ep == 1:
            print(f"  [{tag}] ep {ep:3d} | cls={st['cls_loss']:.3f} im={st['im_loss']:+.3f} | "
                  f"tgt-val={tv:.3f} (sm={sm:.3f}) | {time.time()-t0:.0f}s")
        if stop.early_stop:
            print(f"  [{tag}] early stop at epoch {ep}")
            break
    t.load_state_dict(best_state)

    # AdaBN keep/discard decided on target-VAL only (as in the main notebook)
    _, v0, _, _, _, _ = evaluate(t, tgt_val_loader, cls_criterion, DEVICE, 't')
    t_ab = copy.deepcopy(t)
    adapt_bn(t_ab, tgt_l, DEVICE, which='t', n_pass=3)
    _, v1, _, _, _, _ = evaluate(t_ab, tgt_val_loader, cls_criterion, DEVICE, 't')
    adabn_kept = v1 >= v0
    t = t_ab if adabn_kept else t
    _freeze(t)
    _, te, _, tp, tl, _ = evaluate(t, tgt_test_loader, cls_criterion, DEVICE, 't')
    info = dict(tag=tag, use_im=use_im, best_sm_val=best_sm, val_no_adabn=v0,
                val_adabn=v1, adabn_kept=adabn_kept, tgt_test_acc=te,
                tgt_test_f1_macro=f1_score(tl, tp, average='macro'),
                time_s=time.time() - t0)
    print(f"  [{tag}] done | AdaBN {'kept' if adabn_kept else 'discarded'} "
          f"(val {v0:.4f} -> {v1:.4f}) | target TEST acc={te:.4f}")
    del src_l, tgt_l
    return t, info


def get_teacher(path, use_im=True, tag="teacher"):
    """Load a saved teacher if present, otherwise train it once and save it."""
    if os.path.exists(path):
        t = TeacherNet().to(DEVICE)
        t.load_state_dict(torch.load(path, map_location=DEVICE))
        _freeze(t)
        _, te, _, _, _, _ = evaluate(t, tgt_test_loader, cls_criterion, DEVICE, 't')
        print(f"[{tag}] loaded {path} (md5 {file_md5(path)}) | target TEST acc={te:.4f}")
        return t
    t, info = train_teacher(use_im=use_im, tag=tag)
    torch.save(t.state_dict(), path)
    with open(path.replace('.pth', '_info.json'), 'w') as fh:
        json.dump({k: (float(v) if isinstance(v, (np.floating, float)) else v)
                   for k, v in info.items()}, fh, indent=2)
    print(f"[{tag}] saved -> {path} (md5 {file_md5(path)})")
    return t


def run_student(name, seed, teacher_obj, *, use_se=True, use_vit=True, use_im=True,
                use_pseudo=True, use_kd=True, kd_on_target=True, bg=True,
                model_fn=None, epochs=EPOCHS, teacher_md5="", verbose=True):
    """One student run under the main-notebook protocol. Returns one result row."""
    set_seed(seed)
    src_l, tgt_l = make_train_loaders(seed, bg=bg)
    m = (model_fn() if model_fn is not None
         else MicroBrainNetDA(use_se=use_se, use_vit=use_vit)).to(DEVICE)
    n_params = sum(p.numel() for p in m.parameters())
    T = teacher_obj if use_kd else None
    proj = nn.Linear(m.feature_dim, T.feature_dim).to(DEVICE) if T is not None else None
    params = list(m.parameters()) + (list(proj.parameters()) if proj is not None else [])
    opt = optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    sched = make_scheduler(opt, epochs, WARMUP_EPOCHS)
    stop = EarlyStopping(patience=PATIENCE, min_delta=1e-3)
    ema = EMA(m) if use_pseudo else None

    raw, best_sm, best_state = [], 0.0, copy.deepcopy(m.state_dict())
    pseudo_on, plateau, pseudo_start, last_ps_acc = False, 0, None, float('nan')
    auto_off_fired = 0
    t0, ep = time.time(), 0
    if verbose:
        print(f"\n{'-'*90}\n[{name}] seed={seed} | params={n_params:,} | SE={use_se} ViT={use_vit} "
              f"IM={use_im} PL={use_pseudo} KD={T is not None} KD-on-target={kd_on_target} BG={bg}")

    for ep in range(1, epochs + 1):
        st = train_da_epoch(m, src_l, tgt_l, cls_criterion, opt, DEVICE, ep, epochs, CFG,
                            which='s', teacher=T, kd_proj=proj, ema=ema,
                            pseudo_enabled=pseudo_on, use_coral=False, use_mmd=False,
                            use_im=use_im, use_mcc=False, use_kd=(T is not None),
                            kd_on_target=kd_on_target)
        sched.step()
        _, sv, _, _, _, _ = evaluate(m, src_val_loader, cls_criterion, DEVICE)
        _, tv, _, _, _, _ = evaluate(m, tgt_val_loader, cls_criterion, DEVICE)
        raw.append(tv)
        sm = float(np.mean(raw[-SMOOTH_WINDOW:]))
        if sm > best_sm:
            best_sm, best_state = sm, copy.deepcopy(m.state_dict())
        stop(sm, m)

        if use_pseudo:   # plateau trigger + auto-off, as in the main notebook
            plateau = plateau + 1 if (len(raw) > SMOOTH_WINDOW and sm <= best_sm) else 0
            if (not pseudo_on) and plateau >= PSEUDO_PLATEAU_EPOCHS:
                pseudo_on, pseudo_start = True, ep
            if st['pseudo_used'] > 0:
                last_ps_acc = st['pseudo_acc']
                if PSEUDO_AUTO_OFF and st['pseudo_acc'] < PSEUDO_MIN_MONITOR_ACC:
                    pseudo_on, auto_off_fired = False, auto_off_fired + 1
                    if verbose:
                        print(f"  >> pseudo-labeling AUTO-DISABLED at epoch {ep} "
                              f"(monitor acc {st['pseudo_acc']:.3f} < {PSEUDO_MIN_MONITOR_ACC})")

        if verbose and (ep % 10 == 0 or ep == 1):
            print(f"  ep {ep:3d} | cls={st['cls_loss']:.3f} kd={st['kd_loss']:.3f} "
                  f"im={st['im_loss']:+.3f} | src-val={sv:.3f} tgt-val={tv:.3f} (sm={sm:.3f}) "
                  f"| ps#={st['pseudo_used']} | {time.time()-t0:.0f}s")
        if stop.early_stop:
            break

    m.load_state_dict(best_state)
    _, v0, _, _, _, _ = evaluate(m, tgt_val_loader, cls_criterion, DEVICE)
    m_ab = copy.deepcopy(m)
    adapt_bn(m_ab, tgt_l, DEVICE, which='s', n_pass=3)
    _, v1, _, _, _, _ = evaluate(m_ab, tgt_val_loader, cls_criterion, DEVICE)
    adabn_kept = v1 >= v0
    if adabn_kept:
        m = m_ab
    else:
        del m_ab

    _, sa, _, _, _, _ = evaluate(m, src_test_loader, cls_criterion, DEVICE)
    _, ta, _, tp, tl, tprob = evaluate(m, tgt_test_loader, cls_criterion, DEVICE)
    _, lo, hi = bootstrap_ci(tl, tp, 2000)
    np.savez_compressed(pred_path(name, seed), preds=tp, labels=tl, probs=tprob)

    row = dict(name=name, seed=int(seed), params=int(n_params),
               src_test_acc=float(sa), tgt_val_best_sm=float(best_sm),
               tgt_val_no_adabn=float(v0), tgt_val_adabn=float(v1), adabn_kept=bool(adabn_kept),
               tgt_test_acc=float(ta), tgt_test_f1_macro=float(f1_score(tl, tp, average='macro')),
               ci_lo=float(lo), ci_hi=float(hi), epochs_run=int(ep),
               pseudo_start_epoch=(int(pseudo_start) if pseudo_start else -1),
               last_pseudo_acc=float(last_ps_acc), auto_off_fired=int(auto_off_fired),
               teacher_md5=teacher_md5,
               time_s=float(time.time() - t0))
    if verbose:
        print(f"  -> tgt TEST acc={ta:.4f} macro-F1={row['tgt_test_f1_macro']:.4f} "
              f"[{lo:.3f}, {hi:.3f}] | src TEST={sa:.4f} | AdaBN {'kept' if adabn_kept else 'discarded'} "
              f"| {row['time_s']/60:.1f} min")
    del m, opt, proj, ema, src_l, tgt_l
    torch.cuda.empty_cache()
    return row

In [ ]:
# ============================================================
# SUMMARY: mean +/- std over seeds and paired comparison against a reference row
# ============================================================
# All runs are scored on the same 79 target-test images, so differences between two
# variants are computed per image (paired). The bootstrap resamples test images and
# averages the accuracy difference over the seeds both variants share.

def summarize(reference, order=None, n_boot=5000):
    df = load_runs()
    if df.empty:
        print("no runs yet"); return pd.DataFrame()
    names = order or list(dict.fromkeys(df['name']))
    names = [n for n in names if n in set(df['name'])]
    rng = np.random.RandomState(SEED)

    def preds_for(name):
        out = {}
        for s in sorted(df[df['name'] == name]['seed']):
            p = pred_path(name, s)
            if os.path.exists(p):
                z = np.load(p)
                out[int(s)] = (z['preds'], z['labels'])
        return out

    ref = preds_for(reference)
    rows = []
    for name in names:
        d = df[df['name'] == name]
        r = dict(variant=name, n_seeds=len(d), params=int(d['params'].iloc[0]),
                 acc_mean=100 * d['tgt_test_acc'].mean(), acc_std=100 * d['tgt_test_acc'].std(ddof=1) if len(d) > 1 else float('nan'),
                 f1_mean=100 * d['tgt_test_f1_macro'].mean(), f1_std=100 * d['tgt_test_f1_macro'].std(ddof=1) if len(d) > 1 else float('nan'),
                 src_acc_mean=100 * d['src_test_acc'].mean())
        cur = preds_for(name)
        shared = sorted(set(cur) & set(ref))
        if name != reference and shared:
            corr_v = np.stack([(cur[s][0] == cur[s][1]).astype(float) for s in shared])
            corr_r = np.stack([(ref[s][0] == ref[s][1]).astype(float) for s in shared])
            diff = corr_v - corr_r                       # seeds x images
            n = diff.shape[1]
            boots = np.array([diff[:, rng.randint(0, n, n)].mean() for _ in range(n_boot)])
            r.update(delta_mean=100 * diff.mean(), delta_lo=100 * np.percentile(boots, 2.5),
                     delta_hi=100 * np.percentile(boots, 97.5), delta_seeds=len(shared))
        rows.append(r)
    out = pd.DataFrame(rows)
    out.to_csv(f"{RESULTS_DIR}/summary_vs_{''.join(c if c.isalnum() else '_' for c in reference)}.csv",
               index=False)

    print(f"\n{'Variant':<34}{'Params':>9}{'Seeds':>6}{'Acc (%)':>16}{'Macro-F1 (%)':>16}"
          f"{'Delta vs ' + reference[:12] + ' [95% CI]':>34}")
    print('-' * 115)
    for _, r in out.iterrows():
        acc = f"{r.acc_mean:.2f} +/- {r.acc_std:.2f}" if r.n_seeds > 1 else f"{r.acc_mean:.2f}"
        f1 = f"{r.f1_mean:.2f} +/- {r.f1_std:.2f}" if r.n_seeds > 1 else f"{r.f1_mean:.2f}"
        dl = (f"{r.delta_mean:+.2f} [{r.delta_lo:+.2f}, {r.delta_hi:+.2f}]"
              if 'delta_mean' in r and r.delta_mean == r.delta_mean else "--")
        print(f"{r.variant:<34}{r.params:>9,}{r.n_seeds:>6}{acc:>16}{f1:>16}{dl:>34}")
    print("\nA delta whose 95% CI excludes 0 is supported by the data; one that includes 0 is not.")

    print("\n% ---- LaTeX rows ----")
    for _, r in out.iterrows():
        acc = f"{r.acc_mean:.2f} $\\pm$ {r.acc_std:.2f}" if r.n_seeds > 1 else f"{r.acc_mean:.2f}"
        f1 = f"{r.f1_mean:.2f} $\\pm$ {r.f1_std:.2f}" if r.n_seeds > 1 else f"{r.f1_mean:.2f}"
        dl = (f"{r.delta_mean:+.2f} [{r.delta_lo:+.2f}, {r.delta_hi:+.2f}]"
              if 'delta_mean' in r and r.delta_mean == r.delta_mean else "---")
        print(f"{r.variant} & {r.params:,} & {acc} & {f1} & {dl} \\\\")
    return out

### Experiments

In [ ]:
# ============================================================
# TEACHER: load the saved teacher (no teacher training here)
# ============================================================
# Use the SAME teacher file as the ablation notebook, so every comparison row is distilled from
# the teacher of the ablation table. TEACHER_FILE may be the .pth file itself or a folder that
# contains teacher_final.pth. The notebook stops if the file is not found.
TEACHER_FILE = TEACHER_FILE_DRIVE      # set in the Colab setup cell

TEACHER_PATH = f"{RESULTS_DIR}/teacher_final.pth"

def _find_teacher(p):
    if os.path.isfile(p):
        return p
    hits = sorted(glob.glob(os.path.join(p, "**", "teacher_final.pth"), recursive=True))
    return hits[0] if hits else None

_src = _find_teacher(TEACHER_FILE)
if _src is None and not os.path.exists(TEACHER_PATH):
    raise FileNotFoundError(f"Teacher not found at {TEACHER_FILE}.")
if _src is not None:
    shutil.copy(_src, TEACHER_PATH)

teacher = TeacherNet().to(DEVICE)
teacher.load_state_dict(torch.load(TEACHER_PATH, map_location=DEVICE))
_freeze(teacher)
TEACHER_MD5 = file_md5(TEACHER_PATH)
_, _tv, _, _, _, _ = evaluate(teacher, tgt_val_loader, cls_criterion, DEVICE, 't')
_, _te, _, _, _, _ = evaluate(teacher, tgt_test_loader, cls_criterion, DEVICE, 't')
print(f"teacher loaded | md5 {TEACHER_MD5} | target VAL {_tv:.4f} | target TEST {_te:.4f}")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 115MB/s] 


teacher loaded | md5 f4861e2ba7ce | target VAL 0.8481 | target TEST 0.8481


In [ ]:
# ============================================================
# PARAMETER-MATCHED STUDENTS
# ============================================================
# Every model is trained from scratch at 128x128 with the identical pipeline (KD from the
# same teacher, IM, class-balanced pseudo-labeling, background randomization, AdaBN check).
# Only the architecture changes, and each one is width-scaled to about the same parameter
# count as TinyLeafFormer. ImageNet weights are not used for any student: the proposed
# student has none, and a width-scaled network has no matching checkpoint anyway.
from torchvision.ops import Conv2dNormActivation
from torchvision.models._utils import _make_divisible
from torch.utils.flop_counter import FlopCounterMode

TinyLeafFormer = MicroBrainNetDA          # paper name for the proposed student
TARGET_PARAMS = sum(p.numel() for p in TinyLeafFormer().parameters())


class TVStudent(nn.Module):
    """Wraps a torchvision feature extractor behind extract_features()/classifier."""

    def __init__(self, features, feature_dim, head):
        super().__init__()
        self.features, self.pool = features, nn.AdaptiveAvgPool2d(1)
        self.feature_dim, self.classifier = feature_dim, head

    def extract_features(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


def build_cnn_only(k):
    """TinyLeafFormer without the Micro-ViT, widened so the CNN gets the parameters the
    attention block would have used. This is the key control for the attention block."""
    ch = tuple(_make_divisible(c * k, 8) for c in STUDENT_CHANNELS)
    return MicroBrainNetDA(channels=ch, use_se=True, use_vit=False)


def build_mnv2(w):
    net = tvm.mobilenet_v2(weights=None, width_mult=w, num_classes=NUM_CLASSES)
    last_in = net.features[-1][0].in_channels
    last_out = _make_divisible(1280 * w, 8)       # width multiplier applied to the last 1x1 too
    net.features[-1] = Conv2dNormActivation(last_in, last_out, kernel_size=1,
                                            norm_layer=nn.BatchNorm2d, activation_layer=nn.ReLU6)
    return TVStudent(net.features, last_out,
                     nn.Sequential(nn.Dropout(0.2), nn.Linear(last_out, NUM_CLASSES)))


def build_mnv3s(w):
    net = tvm.mobilenet_v3_small(weights=None, width_mult=w, num_classes=NUM_CLASSES)
    return TVStudent(net.features, net.classifier[0].in_features, net.classifier)


def n_params(m):
    return sum(p.numel() for p in m.parameters())


def match_width(builder, grid):
    best = min(grid, key=lambda k: abs(n_params(builder(k)) - TARGET_PARAMS))
    return float(best), n_params(builder(best))


K_CNN,  P_CNN  = match_width(build_cnn_only, np.round(np.arange(0.80, 2.51, 0.01), 2))
W_MNV2, P_MNV2 = match_width(build_mnv2,     np.round(np.arange(0.10, 1.01, 0.01), 2))
W_MNV3, P_MNV3 = match_width(build_mnv3s,    np.round(np.arange(0.10, 1.01, 0.01), 2))

COMPARE = OrderedDict([
    ('TinyLeafFormer (ours)',            None),
    ('CNN-only, param-matched',          lambda: build_cnn_only(K_CNN)),
    ('MobileNetV2, param-matched',       lambda: build_mnv2(W_MNV2)),
    ('MobileNetV3-Small, param-matched', lambda: build_mnv3s(W_MNV3)),
])
print(f"target params (TinyLeafFormer): {TARGET_PARAMS:,}")
print(f"  CNN-only          k={K_CNN:.2f}  -> {P_CNN:,}")
print(f"  MobileNetV2       w={W_MNV2:.2f}  -> {P_MNV2:,}")
print(f"  MobileNetV3-Small w={W_MNV3:.2f}  -> {P_MNV3:,}")

target params (TinyLeafFormer): 167,131
  CNN-only          k=1.42  -> 169,269
  MobileNetV2       w=0.27  -> 167,083
  MobileNetV3-Small w=0.30  -> 172,211


In [ ]:
# ============================================================
# COST PROFILE (CPU, batch 1): MACs, INT8 weight size, largest layer activation
# ============================================================
# "Largest layer I/O" is the biggest input+output pair of any single layer at INT8, a lower
# bound on the tensor arena for a direct implementation. It does NOT capture how a converter
# lowers an operator (the >6 MB arena of the full TinyLeafFormer graph came from the
# broadcast-based lowering of linear attention), so on-device numbers still come from the
# exported models.

def profile(model):
    model = copy.deepcopy(model).cpu().eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    io, hooks = [], []
    for mod in model.modules():
        if len(list(mod.children())) == 0:
            hooks.append(mod.register_forward_hook(
                lambda m, i, o: io.append((i[0].numel() if (isinstance(i, tuple) and i and torch.is_tensor(i[0])) else 0)
                                          + (o.numel() if torch.is_tensor(o) else 0))))
    with torch.no_grad():
        model(x)
    for h in hooks:
        h.remove()
    p = n_params(model)
    return dict(params=p, mmacs=fc.get_total_flops() / 2 / 1e6,
                int8_weights_kb=p / 1024, largest_layer_io_kb=max(io) / 1024)


cost = OrderedDict()
for nm, fn in COMPARE.items():
    cost[nm] = profile(TinyLeafFormer() if fn is None else fn())
cost_df = pd.DataFrame(cost).T
cost_df.to_csv(f"{RESULTS_DIR}/cost_profile.csv")
print(cost_df.round(1).to_string())

                                    params  mmacs  int8_weights_kb  largest_layer_io_kb
TinyLeafFormer (ours)             167131.0   33.5            163.2                512.0
CNN-only, param-matched           169269.0   56.8            165.3                768.0
MobileNetV2, param-matched        167083.0   10.6            163.2                384.0
MobileNetV3-Small, param-matched  172211.0    3.3            168.2                 80.0


In [ ]:
# ============================================================
# REUSE the ablation 'Full' runs as TinyLeafFormer rows (same teacher only)
# ============================================================
_df = load_runs()
if not _df.empty and 'Full' in set(_df['name']):
    for _, r in _df[_df['name'] == 'Full'].iterrows():
        if r.get('teacher_md5', '') != TEACHER_MD5:
            print(f"  seed {r.seed}: different teacher (md5 {r.get('teacher_md5')}) -> not reused")
            continue
        if is_done('TinyLeafFormer (ours)', int(r.seed)):
            continue
        row = r.to_dict(); row['name'] = 'TinyLeafFormer (ours)'
        src, dst = pred_path('Full', int(r.seed)), pred_path('TinyLeafFormer (ours)', int(r.seed))
        if os.path.exists(src):
            shutil.copy(src, dst)
            append_run(row)
            print(f"  seed {int(r.seed)}: reused ablation 'Full' run")
else:
    print("No ablation runs found; TinyLeafFormer will be trained here.")

No ablation runs found; TinyLeafFormer will be trained here.


In [ ]:
# ============================================================
# RUN  (safe to re-run: finished runs are skipped)
# ============================================================
SEEDS = [42]          # single seed; add 1337, 2024 later if a difference is borderline
todo = [(n, s) for s in SEEDS for n in COMPARE if not is_done(n, s) and n in RUN_ONLY]
print(f"{len(todo)} runs to go")

for name, seed in todo:
    if not time_left_for(estimated_run_hours()):
        print(f"\nStopping before '{name}' seed {seed}: not enough session time left.")
        break
    row = run_student(name, seed, teacher, model_fn=COMPARE[name], teacher_md5=TEACHER_MD5)
    append_run(row)

1 runs to go

------------------------------------------------------------------------------------------
[CNN-only, param-matched] seed=42 | params=169,269 | SE=True ViT=True IM=True PL=True KD=True KD-on-target=True BG=True
  ep   1 | cls=0.966 kd=2.156 im=-0.020 | src-val=0.275 tgt-val=0.354 (sm=0.354) | ps#=0 | 64s
  ep  10 | cls=0.654 kd=1.487 im=-0.218 | src-val=0.957 tgt-val=0.481 (sm=0.542) | ps#=0 | 602s
  ep  20 | cls=0.527 kd=1.179 im=-0.255 | src-val=0.979 tgt-val=0.709 (sm=0.648) | ps#=0 | 1200s
  ep  30 | cls=0.498 kd=1.052 im=-0.272 | src-val=0.982 tgt-val=0.709 (sm=0.694) | ps#=0 | 1796s
  ep  40 | cls=0.449 kd=0.949 im=-0.284 | src-val=0.989 tgt-val=0.722 (sm=0.739) | ps#=0 | 2384s
  ep  50 | cls=0.413 kd=0.816 im=-0.304 | src-val=0.991 tgt-val=0.734 (sm=0.706) | ps#=0 | 2967s
  ep  60 | cls=0.410 kd=0.745 im=-0.326 | src-val=0.998 tgt-val=0.747 (sm=0.747) | ps#=0 | 3553s
  ep  70 | cls=0.392 kd=0.706 im=-0.333 | src-val=0.998 tgt-val=0.759 (sm=0.780) | ps#=17 | 4145s
 